<a href="https://colab.research.google.com/github/dennisquispe/Machine-learning-I/blob/main/Proyecto_de_Regresion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 1: PosGrado - UNAJ

**Integrantes del Grupo**

*   Dennis Saul Quispe Martinez
*   (Integrante 2 - completar)

**Descripción del Dataset**

El dataset “AI Usage & Student Academic Performance” trata sobre el análisis de la relación entre el uso de herramientas de inteligencia artificial, como ChatGPT, y el rendimiento académico de los estudiantes. Su propósito es identificar cómo influye el uso de estas tecnologías en los hábitos de estudio, las calificaciones, las horas dedicadas al aprendizaje y la dependencia de la IA. A través de esta información, se busca descubrir si el uso de herramientas de inteligencia artificial contribuye positivamente al aprendizaje o si, por el contrario, puede generar dependencia y afectar el desarrollo académico de los estudiantes. Esta información permite realizar un análisis exploratorio de datos (EDA), identificar patrones, visualizar tendencias y encontrar posibles relaciones entre el uso de la inteligencia artificial y el desempeño educativo.

*Fuente de datos*

**Kaggle**: AI Usage & Student Academic Performance EDA

https://www.kaggle.com/code/lukhilaksh/ai-usage-student-academic-performance-eda

Titulo: Aprendizaje automático explicable para analizar las relaciones no lineales entre los patrones de uso de inteligencia artificial generativa y la retención de habilidades en estudiantes universitarios

## Diccionario de Variables

| **Variable** | **Descripción** | **Rol en el proyecto** |
|--------------|-----------------|------------------------|
| `Student_ID` | Identificador del estudiante | Identificador (no se usa para modelar) |
| `Major_Category` | Área de la carrera (STEM, Business, Humanities, Medical...) | Predictora (categórica nominal) |
| `Year_of_Study` | Año de estudio (Freshman, Sophomore, Junior, Senior) | Predictora (categórica ordinal) |
| `Pre_Semester_GPA` | Promedio de notas **antes** del semestre (escala 0 a 4) | Predictora (numérica) |
| `Weekly_GenAI_Hours` | Horas semanales de uso de IA generativa | Predictora (numérica) |
| `Primary_Use_Case` | Uso principal de la IA (redacción, ideas, depuración de código...) | Predictora (categórica nominal) |
| `Prompt_Engineering_Skill` | Nivel de habilidad para escribir prompts (Beginner, Intermediate, Advanced) | Predictora (categórica ordinal) |
| `Tool_Diversity` | Cantidad de herramientas de IA distintas que usa | Predictora (numérica) |
| `Paid_Subscription` | Si paga una suscripción de IA (True/False) | Predictora (binaria) |
| `Traditional_Study_Hours` | Horas semanales de estudio tradicional (sin IA) | Predictora (numérica) |
| `Perceived_AI_Dependency` | Dependencia de la IA percibida por el estudiante (escala 1 a 5) | Predictora (numérica ordinal) |
| `Institutional_Policy` | Política de la institución sobre IA (Strict_Ban, Allowed_With_Citation...) | Predictora (categórica nominal) |
| `Anxiety_Level_During_Exams` | Nivel de ansiedad en exámenes (escala 1 a 10) | Predictora (numérica ordinal) |
| `Post_Semester_GPA` | Promedio de notas **después** del semestre (0 a 4) | Resultado (se excluye del modelo) |
| `Skill_Retention_Score` | Puntaje de retención de habilidades (0 a 100) | **Resultado principal** (de aquí sale la variable objetivo) |
| `Burnout_Risk_Level` | Nivel de riesgo de agotamiento (Low, Medium, High) | Resultado (se excluye del modelo) |

> Las escalas se dedujeron de los datos y de los nombres de las variables. Conviene verificarlas con la ficha del dataset en Kaggle.

# Cargar librerías

In [ ]:
# Instalar librerías que no vienen en Colab (solo la primera vez)
!pip install -q catboost imbalanced-learn optuna kagglehub

In [ ]:
# Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from scipy import stats

# Librerías de Kaggle
import kagglehub
import os

# Preprocesamiento y evaluación
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_validate, cross_val_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer, RobustScaler, PowerTransformer
from sklearn.preprocessing import LabelEncoder, KBinsDiscretizer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif, RFE
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, make_scorer
from sklearn.metrics import roc_curve, precision_recall_curve, ConfusionMatrixDisplay
from sklearn.metrics import silhouette_score   # útil para evaluar clustering
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score

# Balanceo de clases (imbalanced-learn)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# ---------------------------------------------------
# Algoritmos de Machine Learning SUPERVISADO
# ---------------------------------------------------
from sklearn.linear_model import LogisticRegression, LassoCV    # Regresión logística y LASSO
from sklearn.svm import SVC                                      # Support Vector Machine (SVM)
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors   # K-Nearest Neighbors (KNN)
from sklearn.tree import DecisionTreeClassifier                  # Decision Tree
from sklearn.ensemble import RandomForestClassifier              # Random Forest
from sklearn.ensemble import AdaBoostClassifier                  # AdaBoost
from sklearn.ensemble import GradientBoostingClassifier          # Gradient Boosting
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB

# Librerías externas (requieren instalación: pip install xgboost lightgbm catboost)
from xgboost import XGBClassifier                                # XGBoost
from lightgbm import LGBMClassifier                              # LightGBM
from catboost import CatBoostClassifier                          # CatBoost

# ---------------------------------------------------
# Algoritmos NO SUPERVISADOS
# ---------------------------------------------------
from sklearn.cluster import KMeans                               # K-Means
from sklearn.cluster import AgglomerativeClustering              # Clustering jerárquico
from scipy.cluster.hierarchy import dendrogram, linkage          # Dendrograma (clustering jerárquico)
from sklearn.cluster import DBSCAN                               # DBSCAN
from sklearn.decomposition import PCA                            # PCA
from sklearn.decomposition import FastICA                        # Análisis de componentes independientes (ICA)

import warnings
warnings.filterwarnings('ignore')

# Semilla para que los resultados se puedan repetir
SEMILLA = 42
sns.set_theme(style="whitegrid")

# Importar data desde Kaggle

In [ ]:
# Descargar el dataset desde Kaggle
path = kagglehub.dataset_download("ranaghulamnabi/ai-usage-and-student-academic-performance-analysis")

print("Dataset descargado en:", path)

# Listar los archivos disponibles para ver el nombre correcto
archivos = os.listdir(path)
print("Archivos en el directorio:", archivos)

# Buscar el primer archivo que termine en .csv
csv_files = [f for f in archivos if f.endswith('.csv')]
if csv_files:
    csv_path = os.path.join(path, csv_files[0])
    print("Cargando archivo:", csv_path)
    df = pd.read_csv(csv_path)
    print("Archivo importado")
else:
    print("No se encontró ningún archivo CSV en la carpeta descargada.")

In [ ]:
df.head()

## FASE 1. Comprensión del negocio

### 1.1. Identificación del problema

Las herramientas de IA generativa (ChatGPT y similares) ya forman parte de la vida universitaria. Muchos estudiantes las usan para resumir, redactar o programar, pero no está claro si **ayudan a aprender o si reemplazan el esfuerzo propio**.

El problema es este: *un uso excesivo de la IA podría generar dependencia y hacer que el estudiante **retenga menos habilidades**, aunque sus notas se vean bien.* Si la universidad detecta a tiempo qué estudiantes están en riesgo, puede intervenir (tutorías, políticas de uso, talleres).

**Pregunta central:** ¿Qué hábitos de uso de IA y de estudio se asocian con una baja retención de habilidades, y es posible predecir qué estudiantes están en riesgo?

**Tipo de problema de datos:**
* **Clasificación binaria (supervisado):** predecir si un estudiante tiene *retención baja* (1) o no (0).
* **Clustering (no supervisado):** descubrir grupos de estudiantes con perfiles de uso parecidos.

### 1.2. Definición de los objetivos

**Objetivo general:** Analizar la relación entre el uso de IA generativa y la retención de habilidades de estudiantes universitarios, y construir un modelo que identifique a los estudiantes con retención baja.

**Objetivos específicos:**
1. Describir el perfil de los estudiantes y sus hábitos de uso de IA (EDA).
2. Medir la relación entre horas de uso de IA, dependencia percibida, estudio tradicional y retención.
3. Preparar los datos (limpieza, transformación, selección de variables, balanceo).
4. Entrenar y comparar modelos supervisados y no supervisados.
5. Evaluar el mejor modelo en rendimiento, generalización, interpretabilidad y equidad.

**Preguntas de investigación:**
* P1: ¿Más horas de uso de IA se asocian con menor retención de habilidades?
* P2: ¿La dependencia percibida de la IA se asocia con menor retención?
* P3: ¿El estudio tradicional compensa el uso de IA?
* P4: ¿Es posible predecir la retención baja con las variables disponibles?
* P5: ¿Hay grupos de estudiantes con perfiles de uso diferentes?

**Variable objetivo (definida por el equipo):** `Retencion_Baja` = 1 si el `Skill_Retention_Score` está dentro del **30 % más bajo** de los estudiantes; 0 en caso contrario.

**Criterio de éxito:** ROC-AUC en el conjunto de prueba **mayor a 0.70** y que el modelo supere claramente al azar (AUC = 0.5).

### 1.3. Formulación de la hipótesis

| Hipótesis | H0 (nula) | H1 (alternativa) |
|-----------|-----------|------------------|
| **H1** | No hay relación entre horas de IA y retención | Más horas de IA → **menor** retención |
| **H2** | No hay relación entre dependencia percibida y retención | Mayor dependencia → **menor** retención |
| **H3** | No hay relación entre estudio tradicional y retención | Más estudio tradicional → **mayor** retención |
| **H4** | La retención es igual en todas las políticas institucionales | La retención **cambia** según la política institucional |
| **H5** | La relación horas de IA–retención es lineal | La relación es **no lineal** (hay un punto donde el efecto cambia) |

Estas hipótesis se contrastan con pruebas estadísticas en la sección 2.2 (C) y se retoman en la sección 5.6.

## FASE 2. Comprensión de los datos

### 2.1. Comprensión de los datos

In [ ]:
# Tamaño y tipos de datos
print("Filas y columnas:", df.shape)
print()
df.info()

In [ ]:
# Agrupamos las variables por tipo (las usaremos en todo el proyecto)
ID          = 'Student_ID'
VARS_NUM    = ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity',
               'Traditional_Study_Hours', 'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams']
VARS_CAT    = ['Major_Category', 'Year_of_Study', 'Primary_Use_Case',
               'Prompt_Engineering_Skill', 'Institutional_Policy', 'Burnout_Risk_Level']
VARS_BOOL   = ['Paid_Subscription']
RESULTADOS  = ['Post_Semester_GPA', 'Skill_Retention_Score']
TODAS_NUM   = VARS_NUM + RESULTADOS

# Verificamos que todas las columnas existan
todas = [ID] + VARS_NUM + VARS_CAT + VARS_BOOL + RESULTADOS
print("Columnas que faltan:", [c for c in todas if c not in df.columns])

# Resumen de cada variable
resumen = pd.DataFrame({'tipo': df.dtypes, 'valores_unicos': df.nunique(), 'nulos': df.isna().sum()})
resumen

**Lectura rápida:** cada fila es un estudiante. Tenemos 6 variables numéricas, 6 categóricas, 1 binaria, 1 identificador y 2 resultados numéricos (`Post_Semester_GPA` y `Skill_Retention_Score`).

### 2.2. Exploración de los datos (EDA)

#### A. Estadística descriptiva

**Análisis univariado** (una variable a la vez)

In [ ]:
# Variables numéricas: media, desviación, mínimo, máximo, cuartiles, asimetría y curtosis
desc = df[TODAS_NUM].describe().T
desc['asimetria'] = df[TODAS_NUM].skew()
desc['curtosis']  = df[TODAS_NUM].kurt()
desc.round(3)

In [ ]:
# Variables categóricas: cuántos estudiantes hay en cada categoría
for col in VARS_CAT + VARS_BOOL:
    tabla = pd.concat([df[col].value_counts(), (df[col].value_counts(normalize=True) * 100).round(1)],
                      axis=1, keys=['n', '%'])
    print(f"--- {col} ---")
    print(tabla)
    print()

**Análisis multivariado** (varias variables a la vez)

In [ ]:
# Promedio de las variables numéricas por cada grupo
print("Promedios por área de carrera")
display(df.groupby('Major_Category')[TODAS_NUM].mean().round(2))

print("Promedios por política institucional")
display(df.groupby('Institutional_Policy')[TODAS_NUM].mean().round(2))

print("Promedios por nivel de riesgo de burnout")
display(df.groupby('Burnout_Risk_Level')[TODAS_NUM].mean().round(2))

**Gráficos univariados**

In [ ]:
# box and whisker plots (cajas y bigotes)
f, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(TODAS_NUM):
    sns.boxplot(y=df[col], ax=axes[i], color="skyblue")
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

**Histogramas**

In [ ]:
# histograms - Seaborn
f, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(TODAS_NUM):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(col)
    axes[i].set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Gráficos de barras para las variables categóricas
cats = VARS_CAT + VARS_BOOL
f, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(cats):
    sns.countplot(x=df[col], ax=axes[i], color="steelblue")
    axes[i].set_title(col)
    axes[i].set_xlabel("")
    axes[i].tick_params(axis='x', rotation=30)
axes[-1].set_visible(False)   # sobra un cuadro
plt.tight_layout()
plt.show()

**Diagrama de dispersión**

In [ ]:
# scatter plot matrix - Seaborn (usamos una muestra de 500 estudiantes para que sea rápido)
muestra = df[VARS_NUM + ['Skill_Retention_Score']].sample(min(500, len(df)), random_state=SEMILLA)
sns.pairplot(muestra, corner=True);

In [ ]:
# scatter plot matrix - Matplotlib
plt.rcParams["figure.figsize"] = [12, 8]
pd.plotting.scatter_matrix(muestra[['Weekly_GenAI_Hours', 'Traditional_Study_Hours',
                                    'Perceived_AI_Dependency', 'Skill_Retention_Score']])
plt.show()
plt.rcParams["figure.figsize"] = [6.4, 4.8]   # volvemos al tamaño normal

#### B. Análisis de distribución

Revisamos si cada variable numérica se parece a una campana normal. Usamos la **asimetría** (si es cercana a 0, es simétrica), la **curtosis** y la prueba de normalidad de D'Agostino (si p > 0.05 se considera normal).

In [ ]:
filas = []
for col in TODAS_NUM:
    x = df[col].dropna()
    p = stats.normaltest(x).pvalue
    filas.append({
        'variable': col,
        'asimetria': round(x.skew(), 3),
        'curtosis': round(x.kurt(), 3),
        'p_normalidad': round(p, 4),
        'forma': 'Asimétrica' if abs(x.skew()) > 0.5 else 'Simétrica',
        'normal (p>0.05)': 'Sí' if p > 0.05 else 'No'
    })
pd.DataFrame(filas)

In [ ]:
# Gráficos Q-Q: si los puntos siguen la línea roja, la variable es cercana a normal
f, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(TODAS_NUM):
    stats.probplot(df[col].dropna(), dist="norm", plot=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

#### C. Análisis de relaciones

In [ ]:
# Correlación de Pearson (relación lineal) y Spearman (relación por orden, más robusta)
f, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(df[TODAS_NUM].corr(method='pearson'),  annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("Pearson")
sns.heatmap(df[TODAS_NUM].corr(method='spearman'), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Spearman")
plt.tight_layout()
plt.show()

In [ ]:
# Relación entre las horas de IA y la retención de habilidades
# Dividimos las horas de IA en 10 grupos y miramos la retención promedio de cada grupo.
# Si la línea no es recta, la relación es no lineal.
grupos = pd.qcut(df['Weekly_GenAI_Hours'].rank(method='first'), q=10, labels=False)
curva = df.groupby(grupos).agg(horas_ia=('Weekly_GenAI_Hours', 'mean'),
                               retencion=('Skill_Retention_Score', 'mean'))

f, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df.sample(min(1500, len(df)), random_state=SEMILLA),
                x='Weekly_GenAI_Hours', y='Skill_Retention_Score', alpha=0.3, ax=axes[0])
axes[0].set_title("Horas de IA vs retención (cada punto es un estudiante)")
axes[1].plot(curva['horas_ia'], curva['retencion'], marker='o')
axes[1].set_title("Retención promedio por grupo de horas de IA")
axes[1].set_xlabel("Horas semanales de IA (promedio del grupo)")
axes[1].set_ylabel("Retención promedio")
plt.tight_layout()
plt.show()

In [ ]:
# Retención según cada variable categórica
f, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()
for i, col in enumerate(VARS_CAT):
    sns.boxplot(data=df, x=col, y='Skill_Retention_Score', ax=axes[i], color="skyblue")
    axes[i].set_title(f"Retención por {col}")
    axes[i].set_xlabel("")
    axes[i].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# Relación entre dos variables categóricas: política institucional vs riesgo de burnout
tabla = pd.crosstab(df['Institutional_Policy'], df['Burnout_Risk_Level'])
display(tabla)

chi2_val, p, gl, esperado = stats.chi2_contingency(tabla)
v_cramer = np.sqrt(chi2_val / (tabla.values.sum() * (min(tabla.shape) - 1)))
print(f"Chi-cuadrado = {chi2_val:.2f}  |  p-valor = {p:.4f}  |  V de Cramer = {v_cramer:.3f}")
print("Hay relación significativa" if p < 0.05 else "No hay relación significativa")

**Contraste de las hipótesis (H1 a H5)**

Usamos Spearman para las relaciones entre números, Kruskal-Wallis para comparar grupos, y para H5 comparamos un ajuste recto contra un ajuste curvo (si la curva explica mejor, la relación es no lineal).

In [ ]:
resultados_hip = []
ALFA = 0.05

# H1, H2, H3: correlación de Spearman con la retención
pruebas = [('H1', 'Weekly_GenAI_Hours',        'negativa'),
           ('H2', 'Perceived_AI_Dependency',   'negativa'),
           ('H3', 'Traditional_Study_Hours',   'positiva')]

for h, var, esperado in pruebas:
    rho, p = stats.spearmanr(df[var], df['Skill_Retention_Score'])
    direccion = 'negativa' if rho < 0 else 'positiva'
    apoya = (p < ALFA) and (direccion == esperado)
    resultados_hip.append({'Hipótesis': h, 'Variable': var, 'Estadístico': f"rho = {rho:.3f}",
                           'p-valor': round(p, 4), 'Esperado': esperado,
                           'Decisión': 'Se rechaza H0 (apoya H1)' if apoya else 'No se rechaza H0'})

# H4: la retención cambia según la política institucional (Kruskal-Wallis)
grupos_pol = [g['Skill_Retention_Score'].values for _, g in df.groupby('Institutional_Policy')]
h_stat, p = stats.kruskal(*grupos_pol)
resultados_hip.append({'Hipótesis': 'H4', 'Variable': 'Institutional_Policy', 'Estadístico': f"H = {h_stat:.2f}",
                       'p-valor': round(p, 4), 'Esperado': 'diferencia',
                       'Decisión': 'Se rechaza H0 (apoya H1)' if p < ALFA else 'No se rechaza H0'})

# H5: relación no lineal. Comparamos R2 de una recta (grado 1) y de una curva (grado 2)
x = df['Weekly_GenAI_Hours'].values
y_ret = df['Skill_Retention_Score'].values
def r2_polinomio(grado):
    coef = np.polyfit(x, y_ret, grado)
    pred = np.polyval(coef, x)
    return 1 - ((y_ret - pred) ** 2).sum() / ((y_ret - y_ret.mean()) ** 2).sum()
r2_lineal, r2_curva = r2_polinomio(1), r2_polinomio(2)
mejora = r2_curva - r2_lineal
resultados_hip.append({'Hipótesis': 'H5', 'Variable': 'Weekly_GenAI_Hours', 'Estadístico': f"R2 recta={r2_lineal:.3f} / curva={r2_curva:.3f}",
                       'p-valor': np.nan, 'Esperado': 'no lineal (mejora R2 > 0.01)',
                       'Decisión': 'Se rechaza H0 (apoya H1)' if mejora > 0.01 else 'No se rechaza H0'})

df_hip = pd.DataFrame(resultados_hip)
df_hip

### 2.3. Evaluación de calidad de datos

In [ ]:
# 1) Valores faltantes
nulos = df.isna().sum()
print("Valores faltantes por columna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "No hay valores faltantes")

# 2) Duplicados
print("\nFilas completamente duplicadas:", df.duplicated().sum())
print("Student_ID repetidos:", df['Student_ID'].duplicated().sum())

# 3) Categorías: revisamos que no haya errores de escritura
print()
for col in VARS_CAT:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

In [ ]:
# 4) Validación de rangos (cuántos valores salen del rango esperado)
reglas = {'Pre_Semester_GPA': (0, 4), 'Post_Semester_GPA': (0, 4), 'Skill_Retention_Score': (0, 100),
          'Weekly_GenAI_Hours': (0, 168), 'Traditional_Study_Hours': (0, 168), 'Tool_Diversity': (0, 50),
          'Perceived_AI_Dependency': (1, 5), 'Anxiety_Level_During_Exams': (1, 10)}
filas = []
for col, (minimo, maximo) in reglas.items():
    fuera = ((df[col] < minimo) | (df[col] > maximo)).sum()
    filas.append({'variable': col, 'rango_esperado': f"{minimo} a {maximo}",
                  'min_real': df[col].min(), 'max_real': df[col].max(), 'fuera_de_rango': fuera})
pd.DataFrame(filas)

In [ ]:
# 5) Valores atípicos (outliers) con la regla del rango intercuartílico (IQR)
filas = []
for col in TODAS_NUM:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    bajo, alto = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < bajo) | (df[col] > alto)).sum()
    filas.append({'variable': col, 'limite_inferior': round(bajo, 2), 'limite_superior': round(alto, 2),
                  'n_outliers': n_out, '% outliers': round(n_out / len(df) * 100, 2)})
pd.DataFrame(filas)

In [ ]:
# 6) Consistencia entre variables y efecto "techo"
horas_totales = df['Weekly_GenAI_Hours'] + df['Traditional_Study_Hours']
print("Estudiantes con más de 100 horas semanales (IA + estudio tradicional):", (horas_totales > 100).sum())
print("Estudiantes sin herramientas de IA (Tool_Diversity = 0) pero con horas de uso:",
      ((df['Tool_Diversity'] == 0) & (df['Weekly_GenAI_Hours'] > 0)).sum())

print("\nEfecto techo (valores pegados al máximo de la escala):")
print("  Pre_Semester_GPA  = 4.0 :", (df['Pre_Semester_GPA'] >= 4.0).sum())
print("  Post_Semester_GPA = 4.0 :", (df['Post_Semester_GPA'] >= 4.0).sum())
print("  Skill_Retention_Score = 100 :", (df['Skill_Retention_Score'] >= 100).sum())

**Conclusión de calidad:** revisar las tablas anteriores. Si no hay faltantes ni duplicados y los rangos son correctos, los datos son de buena calidad. Los puntos a vigilar son (1) los valores pegados al máximo (4.0 en notas y 100 en retención), que indican un *efecto techo*, y (2) los outliers, que no se eliminan sin antes analizarlos (sección 3.2).

### 2.4 Análisis de posible sesgo

In [ ]:
# 1) ¿Todos los grupos están bien representados?
f, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ['Major_Category', 'Year_of_Study', 'Institutional_Policy', 'Paid_Subscription']):
    (df[col].value_counts(normalize=True) * 100).plot(kind='bar', ax=ax, color="steelblue")
    ax.set_title(f"% de estudiantes por {col}")
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Grupos con menos del 10% de los datos
for col in ['Major_Category', 'Year_of_Study', 'Primary_Use_Case', 'Institutional_Policy']:
    porc = df[col].value_counts(normalize=True) * 100
    chicos = porc[porc < 10]
    if len(chicos) > 0:
        print(f"Grupos pequeños (<10%) en {col}: {chicos.round(1).to_dict()}")

In [ ]:
# 2) ¿La retención promedio es muy distinta entre grupos? (puede indicar sesgo en los datos)
for col in ['Major_Category', 'Year_of_Study', 'Institutional_Policy', 'Paid_Subscription']:
    prom = df.groupby(col)['Skill_Retention_Score'].mean().round(2)
    print(f"{col}: diferencia máxima entre grupos = {prom.max() - prom.min():.2f} puntos")
    print(prom.to_dict())
    print()

**Posibles sesgos identificados**

* **Sesgo de representación:** si un grupo (carrera, año o política) tiene pocos estudiantes, el modelo aprende menos sobre él. Se revisa en la sección 5.5.
* **Sesgo de autorreporte:** `Perceived_AI_Dependency` y `Anxiety_Level_During_Exams` son percepciones del propio estudiante, no mediciones objetivas.
* **Efecto techo:** muchas notas en 4.0 y puntajes en 100 limitan la variación y pueden ocultar diferencias entre estudiantes buenos.
* **Origen de los datos:** el dataset no documenta claramente cómo se recolectó. Si fuera simulado o de una sola institución, los resultados **no se pueden generalizar** a otras universidades.
* **Relación no es causalidad:** que dos variables se relacionen no prueba que una cause la otra.
* **Clases desbalanceadas:** la variable objetivo definida (30 % de retención baja) tiene menos casos de una clase. Se trata en la sección 3.9.

## FASE 3. Preparación de los datos

### 3.1. Selección de datos

**Observaciones incluidas:** todos los estudiantes que cumplan los criterios de calidad (valores dentro de rango y con `Skill_Retention_Score` presente).

**Periodo de estudio:** el dataset no tiene fechas. Representa **un semestre** (hay medición antes y después).

**Variables utilizadas (predictoras):** carrera, año de estudio, promedio previo, horas de IA, uso principal, habilidad de prompts, diversidad de herramientas, suscripción paga, horas de estudio tradicional, dependencia percibida, política institucional y ansiedad en exámenes.

**Variables excluidas y por qué:**
* `Student_ID`: es solo un identificador, no aporta información.
* `Skill_Retention_Score`: de aquí se construye la variable objetivo. Dejarla como predictora sería **fuga de información** (el modelo "vería" la respuesta).
* `Post_Semester_GPA` y `Burnout_Risk_Level`: se miden **después** del semestre, igual que la retención. Son resultados, no causas, y no estarían disponibles para predecir con anticipación.

**Variable objetivo:** `Retencion_Baja` = 1 si el puntaje de retención está en el 30 % más bajo.

In [ ]:
# Parámetro que define la variable objetivo (se puede cambiar)
UMBRAL_PERCENTIL = 30    # el 30% de estudiantes con menor retención se marca como "retención baja"

# Qué variables van al modelo y cuáles no
EXCLUIR     = ['Student_ID', 'Skill_Retention_Score', 'Post_Semester_GPA', 'Burnout_Risk_Level']
PREDICTORAS = [c for c in df.columns if c not in EXCLUIR]
print("Variables predictoras:", PREDICTORAS)
print("Variables excluidas  :", EXCLUIR)

# Criterios de inclusión: un valor vacío se acepta (se imputa después); un valor fuera de rango no
def en_rango(serie, minimo, maximo):
    return serie.between(minimo, maximo) | serie.isna()

def cumple_criterios(datos):
    return (datos['Skill_Retention_Score'].between(0, 100)          # el objetivo no puede faltar
            & en_rango(datos['Pre_Semester_GPA'], 0, 4)
            & en_rango(datos['Weekly_GenAI_Hours'], 0, 168)
            & en_rango(datos['Traditional_Study_Hours'], 0, 168)
            & en_rango(datos['Tool_Diversity'], 0, 100))

cumple = cumple_criterios(df)
print(f"\nRegistros que cumplen los criterios: {cumple.sum()} de {len(df)} ({cumple.mean()*100:.2f}%)")

### 3.2. Limpieza de datos

Pasos: formatos, valores faltantes, duplicados, rangos, valores atípicos y creación de la variable objetivo.

> **Importante:** no se eliminan automáticamente los valores atípicos. Primero se revisa si son errores o información real.

**Normalización de formatos**

In [ ]:
# Quitamos espacios sobrantes en los textos
for col in VARS_CAT:
    df[col] = df[col].astype(str).str.strip().replace('nan', np.nan)
print("Formatos de texto normalizados")

**Tratamiento de valores faltantes**

In [ ]:
nulos = df.isna().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "No hay valores faltantes")

# Regla simple: si falta el objetivo, se elimina la fila (no se puede inventar).
# Si falta una predictora numérica -> mediana; si falta una categórica -> moda (valor más frecuente)
df = df.dropna(subset=['Skill_Retention_Score'])
for col in VARS_NUM + ['Post_Semester_GPA']:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
for col in VARS_CAT + VARS_BOOL:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])
print("Faltantes después del tratamiento:", df.isna().sum().sum())

**Eliminación de duplicados**

In [ ]:
n_antes = len(df)
dups = df.duplicated()
print(f"Filas duplicadas detectadas: {dups.sum()} ({dups.sum()/n_antes*100:.2f}%)")

# Eliminamos duplicados y reseteamos el índice
df = df.drop_duplicates().reset_index(drop=True)
print(f"Filas eliminadas: {n_antes - len(df)}")
print(f"Nuevo tamaño: {df.shape}")

**Validación de rangos y corrección de inconsistencias**

In [ ]:
n_antes = len(df)
df = df[cumple_criterios(df)].reset_index(drop=True)
print(f"Filas eliminadas por estar fuera de rango: {n_antes - len(df)}")

# Paid_Subscription: de True/False a 1/0
df['Paid_Subscription'] = df['Paid_Subscription'].astype(int)
print("Tamaño final:", df.shape)

**Identificación y tratamiento de valores atípicos**

In [ ]:
# Revisamos los extremos de las variables de uso. ¿Son errores o casos reales?
for col in ['Weekly_GenAI_Hours', 'Traditional_Study_Hours']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    limite = q3 + 1.5 * (q3 - q1)
    extremos = df[df[col] > limite]
    print(f"{col}: {len(extremos)} valores sobre {limite:.1f} horas (máximo real = {df[col].max():.1f})")

# Una persona no puede usar más de 168 horas semanales (24 x 7). Eso ya se filtró arriba.
# Decisión: los extremos que son posibles (por ejemplo, 40 horas de IA a la semana) son
# información REAL de estudiantes con uso intensivo, y justamente nos interesan.
# Por eso NO se eliminan. Para reducir su efecto usaremos escaladores robustos y Yeo-Johnson (sección 3.3).
print("\nDecisión: se conservan los valores atípicos plausibles.")

**Creación de la variable objetivo y dataset seleccionado**

In [ ]:
# Variable objetivo: 1 = retención baja (30% más bajo), 0 = el resto
umbral = df['Skill_Retention_Score'].quantile(UMBRAL_PERCENTIL / 100)
df['Retencion_Baja'] = (df['Skill_Retention_Score'] < umbral).astype(int)
print(f"Umbral de retención baja: puntaje menor a {umbral:.2f}")
print(df['Retencion_Baja'].value_counts())
print((df['Retencion_Baja'].value_counts(normalize=True) * 100).round(1))

# Dataset seleccionado: solo predictoras + objetivo (y el puntaje original para referencia)
df_sel = df[PREDICTORAS + ['Skill_Retention_Score', 'Retencion_Baja']].copy()
print("\nDataset seleccionado:", df_sel.shape)

### 3.3. Transformación de variables

**Codificación ordinal** (categorías con orden natural)

In [ ]:
df_cod = df_sel.copy()

orden_anio  = {'Freshman': 1, 'Sophomore': 2, 'Junior': 3, 'Senior': 4}
orden_skill = {'Beginner': 1, 'Intermediate': 2, 'Advanced': 3}

df_cod['Year_of_Study']             = df_cod['Year_of_Study'].map(orden_anio)
df_cod['Prompt_Engineering_Skill']  = df_cod['Prompt_Engineering_Skill'].map(orden_skill)

# Si alguna categoría no estaba en el diccionario, aparecería como vacío
print("Vacíos tras codificar:", df_cod[['Year_of_Study', 'Prompt_Engineering_Skill']].isna().sum().to_dict())
df_cod[['Year_of_Study', 'Prompt_Engineering_Skill']].head()

**Label Encoding** (cada categoría pasa a un número; solo ejemplo, no se usa en el modelo porque inventa un orden)

In [ ]:
le = LabelEncoder()
ejemplo = le.fit_transform(df_sel['Major_Category'])
print("Equivalencias:", dict(zip(le.classes_, range(len(le.classes_)))))
print("Primeros valores:", ejemplo[:10])

**One-Hot Encoding** (categorías sin orden: una columna 0/1 por categoría)

In [ ]:
NOMINALES = ['Major_Category', 'Primary_Use_Case', 'Institutional_Policy']
df_cod = pd.get_dummies(df_cod, columns=NOMINALES, drop_first=True, dtype=int)
print("Tamaño tras One-Hot:", df_cod.shape)
print(list(df_cod.columns))

**Agrupación de categorías** (ejemplo: juntar los 4 años de estudio en dos ciclos)

In [ ]:
# Agrupamos años en "Inicial" y "Avanzado" (solo como ejemplo, no se usa en el modelo)
ciclo = df_sel['Year_of_Study'].map({'Freshman': 'Inicial', 'Sophomore': 'Inicial',
                                     'Junior': 'Avanzado', 'Senior': 'Avanzado'})
print(pd.crosstab(ciclo, df_sel['Retencion_Baja'], normalize='index').round(3))

# Función para juntar categorías raras (menos del 5%) en "Otros"
def agrupar_raras(serie, minimo=0.05):
    freq = serie.value_counts(normalize=True)
    raras = freq[freq < minimo].index
    return serie.where(~serie.isin(raras), 'Otros')

print("\nCategorías de Primary_Use_Case antes :", df_sel['Primary_Use_Case'].nunique())
print("Categorías de Primary_Use_Case después:", agrupar_raras(df_sel['Primary_Use_Case']).nunique())

**Transformación logarítmica** (reduce la asimetría de variables con cola larga)

In [ ]:
# log1p = log(1 + x), sirve aunque haya ceros
log_horas = np.log1p(df_cod['Weekly_GenAI_Hours'])

f, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_cod['Weekly_GenAI_Hours'], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title(f"Original (asimetría = {df_cod['Weekly_GenAI_Hours'].skew():.2f})")
sns.histplot(log_horas, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title(f"Logarítmica (asimetría = {log_horas.skew():.2f})")
plt.tight_layout()
plt.show()

**Escalamiento y transformaciones de potencia** (se aplican sobre copias para compararlas)

In [ ]:
# Copias independientes del dataset codificado
X_R       = df_cod.copy()   # MinMax Scaling
X_E       = df_cod.copy()   # Estandarización (Z-score)
X_N       = df_cod.copy()   # Normalización (L2)
X_ROB     = df_cod.copy()   # Robust Scaling
X_T_BOX   = df_cod.copy()   # Box-Cox
X_T_JOH   = df_cod.copy()   # Yeo-Johnson

print("Copias creadas correctamente.")

In [ ]:
# Min-Max: lleva todo al rango 0 a 1
scaler = MinMaxScaler()
X_R[VARS_NUM] = scaler.fit_transform(X_R[VARS_NUM])

print("Rango tras reescalamiento:")
print(X_R[VARS_NUM].describe().loc[["min", "max"]].T.round(3))

In [ ]:
# Z-score: media 0 y desviación 1
scaler_std = StandardScaler()
X_E[VARS_NUM] = scaler_std.fit_transform(X_E[VARS_NUM])

print("Media y Desv. Estándar tras estandarización:")
print(X_E[VARS_NUM].describe().loc[["mean", "std"]].T.round(3))

In [ ]:
# Normalización L2: cada fila (estudiante) queda con longitud 1
normalizer = Normalizer()
X_N[VARS_NUM] = normalizer.fit_transform(X_N[VARS_NUM])

print("Primeras filas normalizadas:")
print(X_N[VARS_NUM].head().round(4))

In [ ]:
# Robust: usa la mediana y el rango intercuartílico, así los outliers molestan menos
robust = RobustScaler()
X_ROB[VARS_NUM] = robust.fit_transform(X_ROB[VARS_NUM])

print("Mediana y IQR tras escalado robusto:")
print(X_ROB[VARS_NUM].describe().loc[["50%"]].T.round(3))

In [ ]:
# Verificar mínimos para seleccionar variables aptas para Box-Cox (solo acepta valores > 0)
mins = df_cod[VARS_NUM].min()
vars_box = mins[mins > 0].index.tolist()
vars_excluidas_box = mins[mins <= 0].index.tolist()

print(f"Variables aptas para Box-Cox (min > 0): {vars_box}")
print(f"Variables EXCLUIDAS de Box-Cox (min <= 0): {vars_excluidas_box}")

transformer_box = PowerTransformer(method="box-cox")
X_T_BOX[vars_box] = transformer_box.fit_transform(X_T_BOX[vars_box])

print("Transformación Box-Cox aplicada exitosamente.")
print(X_T_BOX[vars_box].head().round(4))

In [ ]:
# Yeo-Johnson admite ceros y negativos — aplica a todas las variables numéricas
transformer_yj = PowerTransformer(method="yeo-johnson")
X_T_JOH[VARS_NUM] = transformer_yj.fit_transform(X_T_JOH[VARS_NUM])

print("Transformación Yeo-Johnson aplicada a todas las variables numéricas.")
print(X_T_JOH[VARS_NUM].head().round(4))

In [ ]:
# Comparar distribuciones de una variable representativa (Weekly_GenAI_Hours)
COL = "Weekly_GenAI_Hours"
datasets = {
    "Original"      : df_cod,
    "MinMax"        : X_R,
    "Z-score"       : X_E,
    "Robust"        : X_ROB,
    "Box-Cox"       : X_T_BOX,
    "Yeo-Johnson"   : X_T_JOH,
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, (nombre, dataset) in enumerate(datasets.items()):
    sns.histplot(dataset[COL], bins=20, kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"{COL} — {nombre}")
    axes[i].set_xlabel("")

plt.suptitle(f"Comparativa de Transformaciones — Variable: {COL}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Recorrer todas las variables numéricas
for COL in VARS_NUM:

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()

    for i, (nombre, dataset) in enumerate(datasets.items()):

        # Verificar si la variable existe en el dataset
        if COL in dataset.columns:
            sns.histplot(dataset[COL], bins=20, kde=True, ax=axes[i], color="steelblue")
            axes[i].set_title(f"{COL} — {nombre}")
            axes[i].set_xlabel("")
        else:
            axes[i].set_visible(False)

    plt.suptitle(f"Comparativa de Transformaciones — Variable: {COL}", fontsize=14)
    plt.tight_layout()
    plt.show()

**Decisión sobre las transformaciones**

* Las copias (`X_R`, `X_E`, ...) sirven **solo para comparar** cómo cambia cada distribución.
* Para entrenar los modelos, el escalado (Z-score) se hará **dentro de un Pipeline, después de separar entrenamiento y prueba**. Así el escalador aprende solo de los datos de entrenamiento y no se filtra información de la prueba.
* Los modelos de árboles no necesitan escalado, pero no les afecta, así que se usa el mismo proceso para todos.

**Resultado de esta sección:** `X` (predictoras ya codificadas) e `y` (variable objetivo).

In [ ]:
# Dataset listo para modelar (codificado, todavía sin escalar)
X = df_cod.drop(columns=['Retencion_Baja', 'Skill_Retention_Score'])
y = df_cod['Retencion_Baja']
print("X:", X.shape, "| y:", y.shape)
print("Columnas de X:", list(X.columns))

### 3.4. Imputación

Si faltan datos hay que rellenarlos. Probamos 3 métodos con un experimento: **borramos a propósito el 10 % de `Pre_Semester_GPA`** y vemos qué método recupera mejor el valor real (menor error RMSE = mejor).

In [ ]:
rng = np.random.RandomState(SEMILLA)
col = 'Pre_Semester_GPA'

X_num = df_cod[VARS_NUM].copy()
real = X_num[col].copy()
mask = rng.rand(len(X_num)) < 0.10           # 10% de filas al azar
X_na = X_num.copy()
X_na.loc[mask, col] = np.nan

resultados_imp = {}

# Método 1 y 2: media y mediana
for nombre, imp in {'Media': SimpleImputer(strategy='mean'), 'Mediana': SimpleImputer(strategy='median')}.items():
    X_imp = pd.DataFrame(imp.fit_transform(X_na), columns=X_na.columns, index=X_na.index)
    resultados_imp[nombre] = np.sqrt(((X_imp.loc[mask, col] - real[mask]) ** 2).mean())

# Método 3: KNN (rellena usando los 5 estudiantes más parecidos). Primero estandarizamos.
media, desv = X_na.mean(), X_na.std()
Z = (X_na - media) / desv
Z_imp = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(Z), columns=Z.columns, index=Z.index)
X_knn = Z_imp * desv + media
resultados_imp['KNN (k=5)'] = np.sqrt(((X_knn.loc[mask, col] - real[mask]) ** 2).mean())

tabla_imp = pd.Series(resultados_imp, name='RMSE').round(4).sort_values()
print(tabla_imp)
print(f"\nMejor método en este experimento: {tabla_imp.index[0]}")

**Decisión:** en los pipelines usaremos **imputación por mediana** (`SimpleImputer`). Es simple, robusta a outliers y se ajusta solo con el entrenamiento. Si el dataset no tiene faltantes (como se vio en 2.3), este paso no cambia nada, pero deja el modelo preparado para datos nuevos incompletos.

### 3.5. Discretización

Convertimos variables numéricas en grupos (Bajo / Medio / Alto). Sirve para interpretar mejor y para calcular la entropía en la siguiente sección.

In [ ]:
df_disc = df_sel.copy()

# qcut: grupos con la misma cantidad de estudiantes (terciles). Usamos rank para evitar errores con valores repetidos
df_disc['Uso_IA_nivel'] = pd.qcut(df_disc['Weekly_GenAI_Hours'].rank(method='first'), q=3, labels=['Bajo', 'Medio', 'Alto'])

# cut: rangos que definimos nosotros
df_disc['GPA_previo_nivel'] = pd.cut(df_disc['Pre_Semester_GPA'], bins=[0, 2.5, 3.2, 4.0],
                                     labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)

# KBinsDiscretizer de sklearn (4 grupos con la misma cantidad de estudiantes)
kb = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')
df_disc['Dependencia_grupo'] = kb.fit_transform(df_disc[['Perceived_AI_Dependency']]).ravel()

# ¿Cómo cambia la tasa de retención baja según el nivel de uso de IA y de GPA previo?
f, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['Uso_IA_nivel', 'GPA_previo_nivel']):
    tasa = df_disc.groupby(col, observed=True)['Retencion_Baja'].mean() * 100
    tasa.plot(kind='bar', ax=ax, color="steelblue")
    ax.set_title(f"% de retención baja según {col}")
    ax.set_ylabel("% retención baja")
    ax.set_xlabel("")
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### 3.6. Entropía

La **entropía** mide el desorden o la incertidumbre. Con dos clases, vale 0 si todos son iguales y 1 si hay 50 % y 50 %. La **ganancia de información** dice cuánta incertidumbre del objetivo se elimina al conocer una variable (así eligen sus divisiones los árboles de decisión).

In [ ]:
def entropia(serie):
    p = serie.value_counts(normalize=True)
    return -(p * np.log2(p)).sum()

def ganancia_info(datos, var, objetivo='Retencion_Baja'):
    H_total = entropia(datos[objetivo])
    H_cond = sum(len(g) / len(datos) * entropia(g[objetivo]) for _, g in datos.groupby(var, observed=True))
    return H_total - H_cond

print(f"Entropía de la variable objetivo: {entropia(df_disc['Retencion_Baja']):.4f} (máximo posible = 1)")

# Para las numéricas, usamos 4 grupos por cuantiles
df_ent = df_disc.copy()
for col in VARS_NUM:
    df_ent[col] = pd.qcut(df_ent[col].rank(method='first'), q=4, labels=False)

vars_ent = VARS_NUM + ['Major_Category', 'Year_of_Study', 'Primary_Use_Case', 'Prompt_Engineering_Skill',
                       'Institutional_Policy', 'Paid_Subscription']
tabla_ent = pd.DataFrame({
    'entropia_variable': [entropia(df_ent[c]) for c in vars_ent],
    'ganancia_informacion': [ganancia_info(df_ent, c) for c in vars_ent]
}, index=vars_ent).sort_values('ganancia_informacion', ascending=False)

display(tabla_ent.round(4))
tabla_ent['ganancia_informacion'].plot(kind='barh', figsize=(8, 5), color="steelblue")
plt.title("Ganancia de información sobre Retencion_Baja")
plt.gca().invert_yaxis()
plt.show()

### 3.7. Selección de características (Feature Selection)

Usamos **8 métodos** y cada uno "vota" por las variables que considera importantes. Nos quedamos con las que reciben más votos.

> Para no filtrar información de la prueba, la selección se hace **solo con los datos de entrenamiento** (el mismo 70 % que se usará en la sección 3.8).

In [ ]:
# Parte de entrenamiento (misma división que la sección 3.8)
X_tr_fs, _, y_tr_fs, _ = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMILLA)

# Rellenamos vacíos y escalamos para que los métodos funcionen
X_fs = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X_tr_fs), columns=X.columns, index=X_tr_fs.index)
X_fs_sc = StandardScaler().fit_transform(X_fs)
X_fs_mm = MinMaxScaler().fit_transform(X_fs)      # Chi-cuadrado necesita valores positivos

K = 10   # cuántas variables elige cada método
ranking = pd.DataFrame(index=X.columns)

**Correlación de Pearson y Spearman** (con la variable objetivo)

In [ ]:
ranking['Pearson']  = X_fs.corrwith(y_tr_fs).abs().fillna(0)
ranking['Spearman'] = X_fs.corrwith(y_tr_fs, method='spearman').abs().fillna(0)
ranking[['Pearson', 'Spearman']].sort_values('Pearson', ascending=False).head(10).round(3)

**Chi-cuadrado, ANOVA e Información mutua**

In [ ]:
ranking['Chi2']  = chi2(X_fs_mm, y_tr_fs)[0]
ranking['ANOVA'] = f_classif(X_fs, y_tr_fs)[0]
ranking['MI']    = mutual_info_classif(X_fs, y_tr_fs, random_state=SEMILLA)
ranking[['Chi2', 'ANOVA', 'MI']].sort_values('ANOVA', ascending=False).head(10).round(3)

**RFE** (elimina de a una la variable menos útil), **LASSO** (lleva a cero los coeficientes de variables inútiles) e **importancia por árboles**

In [ ]:
# RFE con regresión logística
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=K).fit(X_fs_sc, y_tr_fs)

# LASSO (con validación cruzada para elegir la penalización)
lasso = LassoCV(cv=5, random_state=SEMILLA).fit(X_fs_sc, y_tr_fs)
ranking['LASSO'] = np.abs(lasso.coef_)

# Importancia de variables con Random Forest
rf_sel = RandomForestClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=-1).fit(X_fs, y_tr_fs)
ranking['RF'] = rf_sel.feature_importances_

print("RFE eligió:", list(X.columns[rfe.support_]))
print("LASSO dejó", (lasso.coef_ != 0).sum(), "variables con coeficiente distinto de cero")

In [ ]:
# Tabla de votos: True = el método eligió esa variable
def top_k(serie, k=K):
    return serie.rank(ascending=False, method='first') <= k

votos = pd.DataFrame({m: top_k(ranking[m]) for m in ['Pearson', 'Spearman', 'Chi2', 'ANOVA', 'MI', 'RF']})
votos['RFE']   = rfe.support_
votos['LASSO'] = lasso.coef_ != 0
votos['Total_votos'] = votos.sum(axis=1)
votos = votos.sort_values('Total_votos', ascending=False)

plt.figure(figsize=(10, 8))
sns.heatmap(votos.drop(columns='Total_votos').astype(int), annot=True, cbar=False, cmap="Blues")
plt.title("Variables elegidas por cada método (1 = elegida)")
plt.show()

# Nos quedamos con las variables que tienen 4 o más votos (de 8). Mínimo 8 variables.
FEATURES = votos[votos['Total_votos'] >= 4].index.tolist()
if len(FEATURES) < 8:
    FEATURES = votos.head(8).index.tolist()

print(f"Variables seleccionadas ({len(FEATURES)}):")
for f_ in FEATURES:
    print(f"  - {f_}  ({votos.loc[f_, 'Total_votos']} votos)")

### 3.8. División de los datos

**70 % entrenamiento y 30 % prueba.** Además usamos **validación cruzada estratificada (5 pliegues)** dentro del entrenamiento para comparar modelos. El conjunto de prueba no se toca hasta la evaluación final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMILLA)

# Nos quedamos solo con las variables seleccionadas
Xtr = X_train[FEATURES]
Xte = X_test[FEATURES]

print(f"Entrenamiento: {Xtr.shape[0]} estudiantes ({Xtr.shape[0]/len(X)*100:.0f}%)")
print(f"Prueba       : {Xte.shape[0]} estudiantes ({Xte.shape[0]/len(X)*100:.0f}%)")
print("\n% de retención baja en cada parte (debe ser parecido gracias a stratify):")
print(f"  Total        : {y.mean()*100:.1f}%")
print(f"  Entrenamiento: {y_train.mean()*100:.1f}%")
print(f"  Prueba       : {y_test.mean()*100:.1f}%")

# Validación cruzada estratificada (5 pliegues)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
for i, (tr, va) in enumerate(cv.split(Xtr, y_train), 1):
    print(f"Pliegue {i}: entrena con {len(tr)}, valida con {len(va)} | % retención baja en validación = {y_train.iloc[va].mean()*100:.1f}%")

### 3.9. Balanceo de clases

Hay menos estudiantes con retención baja (clase 1) que con retención normal (clase 0). Si no se corrige, el modelo tiende a predecir siempre "normal".

> **El balanceo se aplica únicamente a los datos de entrenamiento, nunca a la prueba.**

In [ ]:
# Rellenamos vacíos con la mediana solo para esta demostración
imp_demo = SimpleImputer(strategy='median').fit(Xtr)
Xtr_imp = imp_demo.transform(Xtr)

tecnicas = {
    'Sin balanceo'            : None,
    'SMOTE'                   : SMOTE(random_state=SEMILLA),
    'Random Oversampling'     : RandomOverSampler(random_state=SEMILLA),
    'Random Undersampling'    : RandomUnderSampler(random_state=SEMILLA),
}

filas = []
for nombre, tecnica in tecnicas.items():
    if tecnica is None:
        yb = y_train
    else:
        _, yb = tecnica.fit_resample(Xtr_imp, y_train)
    cuenta = pd.Series(yb).value_counts()
    filas.append({'Técnica': nombre, 'Clase 0': cuenta[0], 'Clase 1': cuenta[1]})
display(pd.DataFrame(filas))

# Class weights: no cambia los datos, solo le da más importancia a la clase minoritaria
pesos = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
print(f"\nClass weights -> clase 0: {pesos[0]:.2f} | clase 1: {pesos[1]:.2f}")

**Decisión:** usaremos **SMOTE** (crea estudiantes sintéticos de la clase minoritaria) dentro del Pipeline. Así se aplica solo a los datos de entrenamiento de cada pliegue de la validación cruzada y nunca contamina la validación ni la prueba.

> Nota: SMOTE interpola entre estudiantes parecidos, por lo que las variables 0/1 (como las de One-Hot) pueden quedar con valores intermedios. Para este proyecto es aceptable; una alternativa más estricta es `SMOTENC`.

**Resultado de la Fase 3:** dataset limpio, codificado, con variables seleccionadas, dividido en entrenamiento/prueba y con un plan de balanceo documentado, listo para entrenar los modelos.

## FASE 4. MODELADO

### 4.1. Machine Learning supervisado

Probamos 12 algoritmos para predecir `Retencion_Baja`: Regresión logística, LDA, k-NN, Árbol de decisión (CART), Naive Bayes, SVM, Random Forest, AdaBoost, Gradient Boosting, XGBoost, LightGBM y CatBoost.

In [ ]:
# Spot-Check Algorithms (supervisado)
models = []
models.append(('LoR', LogisticRegression(max_iter=1000)))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('k-NN', KNeighborsClassifier()))
models.append(('CART', DecisionTreeClassifier(random_state=SEMILLA)))
models.append(('NB', GaussianNB()))
models.append(('SVM', SVC()))
models.append(('RF', RandomForestClassifier(random_state=SEMILLA)))
models.append(('AdaBoost', AdaBoostClassifier(random_state=SEMILLA)))
models.append(('GBM', GradientBoostingClassifier(random_state=SEMILLA)))
models.append(('XGB', XGBClassifier(eval_metric='logloss', random_state=SEMILLA)))
models.append(('LGBM', LGBMClassifier(verbose=-1, random_state=SEMILLA)))
models.append(('CatBoost', CatBoostClassifier(verbose=0, iterations=300, random_seed=SEMILLA)))
print("Modelos a comparar:", [m[0] for m in models])

### 4.2. Machine Learning no supervisado

Aquí **no usamos la variable objetivo**. Buscamos grupos de estudiantes parecidos (K-Means, jerárquico, DBSCAN) y reducimos dimensiones (PCA, ICA). Usamos todas las variables predictoras, sin selección, porque el clustering no debe depender del objetivo.

In [ ]:
# Escalado de datos (recomendado para estos algoritmos)
X_cl = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cl)   # X = tus variables, sin la columna objetivo

# Muestra de 3000 estudiantes para que los algoritmos pesados sean rápidos
rng = np.random.RandomState(SEMILLA)
idx = rng.choice(len(X_scaled), size=min(3000, len(X_scaled)), replace=False)
X_s = X_scaled[idx]
y_s = y.values[idx]
print("Datos para clustering:", X_s.shape)

**¿Cuántos grupos (k)?** Probamos de 2 a 8 con el método del codo (inercia) y el índice de silueta.

In [ ]:
ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=7, n_init=10).fit(X_s)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_s, km.labels_))

f, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inercias, marker='o'); axes[0].set_title("Método del codo (inercia)"); axes[0].set_xlabel("k")
axes[1].plot(list(ks), siluetas, marker='o', color='darkorange'); axes[1].set_title("Índice de silueta"); axes[1].set_xlabel("k")
plt.tight_layout(); plt.show()

K_OPT = list(ks)[int(np.argmax(siluetas))]
print(f"k elegido por mayor silueta: {K_OPT}")

In [ ]:
# Algoritmos de clustering
# Para DBSCAN, elegimos eps con la distancia al 5.º vecino más cercano (percentil 90)
dist, _ = NearestNeighbors(n_neighbors=5).fit(X_s).kneighbors(X_s)
eps_auto = float(np.percentile(dist[:, -1], 90))
print(f"eps para DBSCAN: {eps_auto:.2f}")

clustering_models = []
clustering_models.append(('K-Means', KMeans(n_clusters=K_OPT, random_state=7, n_init=10)))
clustering_models.append(('Jerárquico', AgglomerativeClustering(n_clusters=K_OPT, linkage='ward')))
clustering_models.append(('DBSCAN', DBSCAN(eps=eps_auto, min_samples=5)))

etiquetas = {}
for nombre, modelo in clustering_models:
    etiquetas[nombre] = modelo.fit_predict(X_s)
    grupos = pd.Series(etiquetas[nombre]).value_counts().sort_index().to_dict()
    print(f"{nombre}: estudiantes por grupo = {grupos}")

In [ ]:
# Dendrograma del clustering jerárquico (muestra de 200 estudiantes para que se lea bien)
Z = linkage(X_s[:200], method='ward')
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=90)
plt.title("Dendrograma (Ward)")
plt.xlabel("Grupos agrupados")
plt.ylabel("Distancia")
plt.show()

**PCA** (resume las variables en pocas componentes) y **ICA** (busca componentes independientes)

In [ ]:
# PCA sobre todos los datos
pca = PCA(random_state=SEMILLA).fit(X_scaled)
acum = np.cumsum(pca.explained_variance_ratio_)
n80 = int(np.argmax(acum >= 0.80)) + 1
print(f"Componentes necesarias para explicar el 80% de la varianza: {n80} de {X_scaled.shape[1]}")

pca2 = PCA(n_components=2, random_state=SEMILLA).fit(X_scaled)
P = pca2.transform(X_s)

f, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(range(1, len(acum) + 1), acum, marker='o'); axes[0].axhline(0.8, color='red', linestyle='--')
axes[0].set_title("Varianza explicada acumulada (PCA)"); axes[0].set_xlabel("Componentes")
axes[1].scatter(P[:, 0], P[:, 1], c=y_s, cmap='coolwarm', alpha=0.5, s=10)
axes[1].set_title("PCA coloreado por Retencion_Baja")
axes[2].scatter(P[:, 0], P[:, 1], c=etiquetas['K-Means'], cmap='viridis', alpha=0.5, s=10)
axes[2].set_title("PCA coloreado por grupo de K-Means")
plt.tight_layout(); plt.show()

In [ ]:
# ICA con 2 componentes
ica = FastICA(n_components=2, random_state=SEMILLA, max_iter=1000)
I = ica.fit_transform(X_s)

plt.figure(figsize=(6, 5))
plt.scatter(I[:, 0], I[:, 1], c=y_s, cmap='coolwarm', alpha=0.5, s=10)
plt.title("ICA coloreado por Retencion_Baja")
plt.xlabel("Componente 1"); plt.ylabel("Componente 2")
plt.show()

### 4.3. Diseño experimental

Para que la comparación sea justa, **todos los modelos usan exactamente lo mismo**:

| Elemento | Decisión |
|----------|----------|
| Conjunto de entrenamiento | 70 % de los estudiantes (estratificado) |
| Conjunto de validación | Cada pliegue de la validación cruzada dentro del entrenamiento |
| Conjunto de prueba | 30 % que no se usa hasta la evaluación final |
| Validación cruzada | Estratificada de 5 pliegues, con las mismas particiones para todos |
| Estratificación | Se mantiene el % de retención baja en cada parte |
| Número de iteraciones | 5 pliegues; 10 combinaciones en Random Search; 15 pruebas en Optuna |
| Semilla aleatoria | `SEMILLA = 42` en todo el proyecto |
| Preprocesamiento | Imputar (mediana) → Estandarizar → SMOTE → Modelo (todo dentro del Pipeline) |
| Métricas | Accuracy, Precision, Recall, Specificity, F1, ROC-AUC y PR-AUC |
| Métrica principal | **ROC-AUC** (no depende del umbral y funciona con clases desbalanceadas) |

In [ ]:
# Pipeline: cada paso se ajusta SOLO con los datos de entrenamiento de cada pliegue
def crear_pipeline(modelo, balanceo=True):
    pasos = [('imputar', SimpleImputer(strategy='median')),
             ('escalar', StandardScaler())]
    if balanceo:
        pasos.append(('smote', SMOTE(random_state=SEMILLA)))
    pasos.append(('modelo', modelo))
    return ImbPipeline(pasos)

# Métricas (la especificidad es el recall de la clase 0)
scoring = {
    'accuracy'   : 'accuracy',
    'precision'  : 'precision',
    'recall'     : 'recall',
    'specificity': make_scorer(recall_score, pos_label=0),
    'f1'         : 'f1',
    'roc_auc'    : 'roc_auc',
    'pr_auc'     : 'average_precision',
}

# La misma partición (cv) para todos los modelos
print("Validación cruzada:", cv)
print("Métricas:", list(scoring.keys()))

### 4.4. Entrenamiento de los modelos

Para cada modelo se documenta: algoritmo, variables predictoras, variable objetivo, parámetros iniciales, configuración, tiempo de ejecución y resultados.

In [ ]:
resultados = []            # tabla de resultados
modelos_entrenados = {}   # modelos ya entrenados con todo el entrenamiento
auc_pliegues = {}         # AUC de cada pliegue (para el gráfico)
documentacion = []        # ficha de cada modelo

def evaluar_cv(nombre, pipe):
    """Valida con cross_validate, entrena con todo el entrenamiento y guarda los resultados."""
    t0 = time.time()
    cvres = cross_validate(pipe, Xtr, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
    tiempo_cv = time.time() - t0

    t1 = time.time()
    pipe.fit(Xtr, y_train)
    tiempo_fit = time.time() - t1

    resultados.append({
        'Modelo'       : nombre,
        'Accuracy'     : cvres['test_accuracy'].mean(),
        'Precision'    : cvres['test_precision'].mean(),
        'Recall'       : cvres['test_recall'].mean(),
        'Specificity'  : cvres['test_specificity'].mean(),
        'F1'           : cvres['test_f1'].mean(),
        'ROC-AUC'      : cvres['test_roc_auc'].mean(),
        'PR-AUC'       : cvres['test_pr_auc'].mean(),
        'AUC_std'      : cvres['test_roc_auc'].std(),
        'AUC_train'    : cvres['train_roc_auc'].mean(),
        'Tiempo_CV_s'  : tiempo_cv,
        'Tiempo_fit_s' : tiempo_fit,
    })
    auc_pliegues[nombre] = cvres['test_roc_auc']
    modelos_entrenados[nombre] = pipe
    return pipe

def descripcion_parametros(modelo):
    if 'CatBoost' in type(modelo).__name__:
        return str(modelo.get_params())
    return repr(modelo)

In [ ]:
for nombre, modelo in models:
    pipe = crear_pipeline(modelo)
    evaluar_cv(nombre, pipe)
    ultimo = resultados[-1]
    print(f"{nombre:9s} | AUC validación = {ultimo['ROC-AUC']:.3f} (+/- {ultimo['AUC_std']:.3f}) | tiempo = {ultimo['Tiempo_CV_s']:.1f} s")
    documentacion.append({
        'Algoritmo'            : type(modelo).__name__,
        'Variables predictoras': len(FEATURES),
        'Variable objetivo'    : 'Retencion_Baja',
        'Parámetros iniciales' : descripcion_parametros(modelo),
        'Configuración'        : 'Imputar mediana + Z-score + SMOTE; CV estratificada 5 pliegues; semilla 42',
        'Tiempo CV (s)'        : round(ultimo['Tiempo_CV_s'], 1),
        'ROC-AUC validación'   : round(ultimo['ROC-AUC'], 3),
    })

In [ ]:
# Ficha de documentación de cada modelo
pd.set_option('display.max_colwidth', 80)
print("Variables predictoras usadas:", FEATURES)
pd.DataFrame(documentacion)

### 4.5. Optimización de hiperparámetros

Los hiperparámetros son "perillas" del algoritmo que se ajustan a mano. Probamos 3 formas de búsqueda sobre 3 modelos:

* **Grid Search** (prueba todas las combinaciones) → Regresión logística.
* **Random Search** (prueba combinaciones al azar) → Random Forest.
* **Optimización bayesiana con Optuna** (aprende de las pruebas anteriores) → LightGBM.

Todas usan la misma validación cruzada y la métrica ROC-AUC.

In [ ]:
# --- 1) Grid Search: Regresión logística ---
grid = GridSearchCV(crear_pipeline(LogisticRegression(max_iter=1000)),
                    param_grid={'modelo__C': [0.01, 0.1, 1, 10, 100]},
                    scoring='roc_auc', cv=cv, n_jobs=-1)
grid.fit(Xtr, y_train)
print("Grid Search  - mejores parámetros:", grid.best_params_, "| AUC =", round(grid.best_score_, 4))

In [ ]:
# --- 2) Random Search: Random Forest ---
param_rf = {
    'modelo__n_estimators'    : [100, 200, 300, 500],
    'modelo__max_depth'       : [None, 5, 10, 15, 20],
    'modelo__min_samples_leaf': [1, 2, 5, 10],
    'modelo__max_features'    : ['sqrt', 'log2', 0.5],
}
rand = RandomizedSearchCV(crear_pipeline(RandomForestClassifier(random_state=SEMILLA, n_jobs=-1)),
                          param_distributions=param_rf, n_iter=10, scoring='roc_auc',
                          cv=cv, random_state=SEMILLA, n_jobs=1)
rand.fit(Xtr, y_train)
print("Random Search - mejores parámetros:", rand.best_params_, "| AUC =", round(rand.best_score_, 4))

In [ ]:
# --- 3) Optimización bayesiana con Optuna: LightGBM ---
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    modelo = LGBMClassifier(
        n_estimators     = trial.suggest_int('n_estimators', 100, 400),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        num_leaves       = trial.suggest_int('num_leaves', 15, 63),
        min_child_samples= trial.suggest_int('min_child_samples', 10, 50),
        verbose=-1, random_state=SEMILLA, n_jobs=1)
    return cross_val_score(crear_pipeline(modelo), Xtr, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

estudio = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEMILLA))
estudio.optimize(objetivo, n_trials=15)
print("Optuna - mejores parámetros:", estudio.best_params, "| AUC =", round(estudio.best_value, 4))

In [ ]:
# Evaluamos los 3 modelos optimizados con las mismas métricas y los agregamos a la tabla de resultados
pipe_lor_opt  = crear_pipeline(LogisticRegression(max_iter=1000, C=grid.best_params_['modelo__C']))
pipe_rf_opt   = crear_pipeline(RandomForestClassifier(random_state=SEMILLA, n_jobs=-1,
                               **{k.replace('modelo__', ''): v for k, v in rand.best_params_.items()}))
pipe_lgbm_opt = crear_pipeline(LGBMClassifier(verbose=-1, random_state=SEMILLA, n_jobs=1, **estudio.best_params))

evaluar_cv('LoR (Grid)',     pipe_lor_opt)
evaluar_cv('RF (Random)',    pipe_rf_opt)
evaluar_cv('LGBM (Optuna)',  pipe_lgbm_opt)

res_df = pd.DataFrame(resultados).set_index('Modelo')
mejora = pd.DataFrame({
    'AUC antes' : [res_df.loc['LoR', 'ROC-AUC'], res_df.loc['RF', 'ROC-AUC'], res_df.loc['LGBM', 'ROC-AUC']],
    'AUC después': [res_df.loc['LoR (Grid)', 'ROC-AUC'], res_df.loc['RF (Random)', 'ROC-AUC'], res_df.loc['LGBM (Optuna)', 'ROC-AUC']],
}, index=['Regresión logística (Grid)', 'Random Forest (Random)', 'LightGBM (Optuna)'])
mejora['Mejora'] = mejora['AUC después'] - mejora['AUC antes']
mejora.round(4)

> Como la búsqueda se hace sobre la misma validación cruzada, la mejora puede ser algo optimista. Por eso se confirma después con el conjunto de prueba (sección 5.1).

### 4.6. Comparación inicial de modelos

Se compara cada modelo en **rendimiento** (AUC), **estabilidad** (desviación entre pliegues), **generalización** (diferencia entre entrenamiento y validación) y **complejidad** (tiempo). **No se elige automáticamente el de mayor exactitud (accuracy).**

In [ ]:
comparacion = pd.DataFrame(resultados).set_index('Modelo')
comparacion['Brecha_AUC (train-val)'] = comparacion['AUC_train'] - comparacion['ROC-AUC']
comparacion = comparacion.sort_values('ROC-AUC', ascending=False)
display(comparacion[['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1', 'ROC-AUC', 'PR-AUC',
                     'AUC_std', 'Brecha_AUC (train-val)', 'Tiempo_CV_s']].round(3))

# Si siempre se predijera "retención normal", el accuracy sería este (línea base):
base = 1 - y_train.mean()
print(f"\nAccuracy de la línea base (predecir siempre la clase mayoritaria): {base:.3f}")

In [ ]:
f, axes = plt.subplots(1, 2, figsize=(16, 6))

# AUC de cada pliegue por modelo
orden = comparacion.index.tolist()
axes[0].boxplot([auc_pliegues[m] for m in orden], vert=False)
axes[0].set_yticklabels(orden)
axes[0].invert_yaxis()
axes[0].set_title("ROC-AUC por pliegue de validación")
axes[0].set_xlabel("ROC-AUC")

# Rendimiento vs tiempo
axes[1].scatter(comparacion['Tiempo_CV_s'], comparacion['ROC-AUC'])
for m, fila in comparacion.iterrows():
    axes[1].annotate(m, (fila['Tiempo_CV_s'], fila['ROC-AUC']), fontsize=8)
axes[1].set_xlabel("Tiempo de validación cruzada (s)")
axes[1].set_ylabel("ROC-AUC")
axes[1].set_title("Rendimiento vs complejidad")
plt.tight_layout()
plt.show()

print("Top 3 por ROC-AUC en validación:", comparacion.head(3).index.tolist())

## FASE 5. Evaluación (Evaluation)

### 5.1. Evaluación del rendimiento predictivo

#### A. Para clasificación (conjunto de PRUEBA)

In [ ]:
def puntuaciones(modelo, X_):
    """Devuelve la probabilidad (o la puntuación) de la clase 1."""
    if hasattr(modelo, 'predict_proba'):
        return modelo.predict_proba(X_)[:, 1]
    return modelo.decision_function(X_)

filas = []
pred_test, score_test = {}, {}
for nombre, pipe in modelos_entrenados.items():
    y_pred = pipe.predict(Xte)
    sc = puntuaciones(pipe, Xte)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    pred_test[nombre], score_test[nombre] = y_pred, sc
    filas.append({
        'Modelo'     : nombre,
        'Accuracy'   : accuracy_score(y_test, y_pred),
        'Precision'  : precision_score(y_test, y_pred),
        'Recall'     : recall_score(y_test, y_pred),
        'Specificity': tn / (tn + fp),
        'F1'         : f1_score(y_test, y_pred),
        'ROC-AUC'    : roc_auc_score(y_test, sc),
        'PR-AUC'     : average_precision_score(y_test, sc),
    })

tabla_test = pd.DataFrame(filas).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
tabla_test.round(3)

In [ ]:
# Matrices de confusión de los 3 mejores modelos
top3 = tabla_test['Modelo'].head(3).tolist()
f, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, nombre in zip(axes, top3):
    ConfusionMatrixDisplay.from_predictions(y_test, pred_test[nombre], ax=ax, cmap='Blues', colorbar=False,
                                            display_labels=['Normal', 'Retención baja'])
    ax.set_title(nombre)
plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC y Precision-Recall de los 5 mejores modelos
top5 = tabla_test['Modelo'].head(5).tolist()
f, axes = plt.subplots(1, 2, figsize=(14, 5))
for nombre in top5:
    fpr, tpr, _ = roc_curve(y_test, score_test[nombre])
    axes[0].plot(fpr, tpr, label=f"{nombre} ({roc_auc_score(y_test, score_test[nombre]):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, score_test[nombre])
    axes[1].plot(rec, prec, label=nombre)
axes[0].plot([0, 1], [0, 1], 'k--', label='Azar')
axes[0].set_title("Curva ROC"); axes[0].set_xlabel("1 - Especificidad"); axes[0].set_ylabel("Sensibilidad"); axes[0].legend()
axes[1].axhline(y_test.mean(), color='k', linestyle='--', label='Azar')
axes[1].set_title("Curva Precision-Recall"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
plt.tight_layout()
plt.show()

#### B. Para clustering

In [ ]:
filas = []
for nombre, etiq in etiquetas.items():
    valido = etiq != -1                      # DBSCAN marca el ruido con -1
    n_grupos = len(set(etiq[valido]))
    fila = {'Algoritmo': nombre, 'N° grupos': n_grupos, 'Ruido (%)': round((~valido).mean() * 100, 1)}
    if n_grupos >= 2:
        fila['Silhouette (mayor = mejor)']        = silhouette_score(X_s[valido], etiq[valido])
        fila['Davies-Bouldin (menor = mejor)']    = davies_bouldin_score(X_s[valido], etiq[valido])
        fila['Calinski-Harabasz (mayor = mejor)'] = calinski_harabasz_score(X_s[valido], etiq[valido])
    filas.append(fila)
display(pd.DataFrame(filas).round(3))

In [ ]:
# Estabilidad de K-Means: ajustamos con el 80% de los datos al azar y comparamos con los grupos originales
# (ARI = 1 significa grupos idénticos; cerca de 0 significa grupos al azar)
referencia = etiquetas['K-Means']
aris = []
for semilla in range(5):
    r = np.random.RandomState(semilla)
    sub = r.choice(len(X_s), size=int(len(X_s) * 0.8), replace=False)
    km = KMeans(n_clusters=K_OPT, random_state=semilla, n_init=10).fit(X_s[sub])
    aris.append(adjusted_rand_score(referencia, km.predict(X_s)))
print("ARI de estabilidad (5 repeticiones):", np.round(aris, 3))
print(f"Estabilidad promedio = {np.mean(aris):.3f}")

In [ ]:
# Perfil de cada grupo de K-Means: ¿qué los diferencia y qué tasa de retención baja tiene cada uno?
perfil = X_cl.iloc[idx][VARS_NUM].copy()
perfil['Retencion_Baja (%)'] = y_s * 100
perfil['Grupo'] = etiquetas['K-Means']
display(perfil.groupby('Grupo').mean().round(2).assign(N=perfil.groupby('Grupo').size()))

### 5.2. Comparación estadística de modelos

¿La diferencia entre dos modelos es real o puede ser casualidad de la muestra? Usamos el **conjunto de prueba** con:

* **Bootstrap pareado:** repetimos 500 veces el cálculo del AUC con muestras al azar de la prueba (las mismas para todos los modelos) y obtenemos un **intervalo de confianza del 95 %**.
* **Prueba de McNemar:** compara los aciertos y errores del mejor modelo contra otro, estudiante por estudiante.

> No se tratan los resultados de los pliegues de la validación cruzada como observaciones independientes, porque comparten datos de entrenamiento.

In [ ]:
B = 500
rng = np.random.RandomState(SEMILLA)
y_arr = y_test.values
n = len(y_arr)

aucs_boot = {m: [] for m in top5}
for b in range(B):
    ib = rng.randint(0, n, n)                    # mismos estudiantes para todos los modelos
    if len(np.unique(y_arr[ib])) < 2:
        continue
    for m in top5:
        aucs_boot[m].append(roc_auc_score(y_arr[ib], score_test[m][ib]))

mejor = top5[0]
filas = []
for m in top5:
    a = np.array(aucs_boot[m])
    dif = np.array(aucs_boot[mejor]) - a
    filas.append({'Modelo': m,
                  'AUC prueba': round(roc_auc_score(y_test, score_test[m]), 4),
                  'IC 95% inferior': round(np.percentile(a, 2.5), 4),
                  'IC 95% superior': round(np.percentile(a, 97.5), 4),
                  f'Dif. con {mejor} (IC 95%)': '-' if m == mejor else f"[{np.percentile(dif, 2.5):.4f}, {np.percentile(dif, 97.5):.4f}]",
                  'Diferencia significativa': '-' if m == mejor else ('Sí' if (np.percentile(dif, 2.5) > 0 or np.percentile(dif, 97.5) < 0) else 'No')})
display(pd.DataFrame(filas))
print("Si el intervalo de la diferencia incluye el 0, no hay evidencia de que el mejor modelo sea realmente mejor.")

In [ ]:
# McNemar: el mejor modelo contra cada uno de los otros
filas = []
acierto_mejor = (pred_test[mejor] == y_arr)
for m in top5[1:]:
    acierto_otro = (pred_test[m] == y_arr)
    b = int((acierto_mejor & ~acierto_otro).sum())    # acierta el mejor, falla el otro
    c = int((~acierto_mejor & acierto_otro).sum())    # falla el mejor, acierta el otro
    p = 1.0 if (b + c) == 0 else stats.binomtest(b, b + c, 0.5).pvalue
    filas.append({'Comparación': f"{mejor} vs {m}", 'Solo acierta el mejor': b, 'Solo acierta el otro': c,
                  'p-valor': round(p, 4), 'Diferencia': 'Significativa' if p < 0.05 else 'No significativa'})
pd.DataFrame(filas)

### 5.3. Evaluación de generalización

Comprobamos si el modelo funciona igual con datos que no vio. Reglas usadas:

* **Subajuste:** AUC de validación menor a 0.60 (el modelo no aprende lo suficiente).
* **Sobreajuste:** AUC de entrenamiento supera al de validación por más de 0.10.
* **Generaliza bien:** en los demás casos.

**Validación externa:** no se dispone de otra muestra independiente (otra universidad u otro semestre). Es una limitación y se recomienda como trabajo futuro.

In [ ]:
gen = comparacion[['AUC_train', 'ROC-AUC']].rename(columns={'AUC_train': 'AUC entrenamiento', 'ROC-AUC': 'AUC validación (CV)'})
gen['AUC prueba'] = tabla_test.set_index('Modelo')['ROC-AUC']
gen['Brecha entren.-prueba'] = gen['AUC entrenamiento'] - gen['AUC prueba']
gen['Brecha validación-prueba'] = (gen['AUC validación (CV)'] - gen['AUC prueba']).abs()

def diagnostico(fila):
    if fila['AUC validación (CV)'] < 0.60:
        return 'Subajuste'
    if fila['AUC entrenamiento'] - fila['AUC validación (CV)'] > 0.10:
        return 'Sobreajuste'
    return 'Generaliza bien'
gen['Diagnóstico'] = gen.apply(diagnostico, axis=1)
gen.round(3)

In [ ]:
# Curva de aprendizaje del mejor modelo: ¿mejora con más datos? ¿hay mucha distancia entre las curvas?
mejor_modelo = tabla_test.loc[0, 'Modelo']
tamanios, tr_sc, va_sc = learning_curve(modelos_entrenados[mejor_modelo], Xtr, y_train, cv=cv, scoring='roc_auc',
                                        train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1)
plt.figure(figsize=(7, 5))
plt.plot(tamanios, tr_sc.mean(axis=1), marker='o', label='Entrenamiento')
plt.plot(tamanios, va_sc.mean(axis=1), marker='o', label='Validación')
plt.title(f"Curva de aprendizaje - {mejor_modelo}")
plt.xlabel("Cantidad de estudiantes de entrenamiento"); plt.ylabel("ROC-AUC")
plt.legend(); plt.show()

In [ ]:
# Robustez: ¿qué pasa si los datos de prueba tienen "ruido" (error de medición del 10%)?
rng = np.random.RandomState(SEMILLA)
Xte_ruido = Xte.copy().astype(float)
for col in [c for c in VARS_NUM if c in FEATURES]:
    Xte_ruido[col] = Xte_ruido[col] + rng.normal(0, 0.10 * Xte[col].std(), len(Xte))

filas = []
for m in top5:
    auc_normal = roc_auc_score(y_test, score_test[m])
    auc_ruido = roc_auc_score(y_test, puntuaciones(modelos_entrenados[m], Xte_ruido))
    filas.append({'Modelo': m, 'AUC normal': auc_normal, 'AUC con ruido': auc_ruido, 'Pérdida': auc_normal - auc_ruido})
pd.DataFrame(filas).round(4)

### 5.4. Evaluación de interpretabilidad

Vemos **qué variables importan**, **en qué dirección** empujan la predicción, si los efectos son **lineales o no**, y cómo se comporta el modelo con **estudiantes individuales**.

In [ ]:
# 1) Variables más importantes: importancia por permutación en el conjunto de prueba
# (se desordena una variable y se mira cuánto empeora el AUC; más caída = más importante)
modelo_int = modelos_entrenados[mejor_modelo]
perm = permutation_importance(modelo_int, Xte, y_test, scoring='roc_auc', n_repeats=5, random_state=SEMILLA, n_jobs=-1)
imp = pd.DataFrame({'variable': FEATURES, 'importancia': perm.importances_mean, 'desv': perm.importances_std}) \
        .sort_values('importancia', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(imp['variable'], imp['importancia'], xerr=imp['desv'], color="steelblue")
plt.gca().invert_yaxis()
plt.title(f"Importancia por permutación - {mejor_modelo}")
plt.xlabel("Caída del AUC al desordenar la variable")
plt.show()

vars_top = imp['variable'].head(5).tolist()
print("Variables más importantes:", vars_top)

In [ ]:
# 2) Dirección de los efectos: coeficientes de la regresión logística (variables estandarizadas)
# Positivo = aumenta la probabilidad de retención baja | Negativo = la disminuye
coefs = pd.Series(modelos_entrenados['LoR'].named_steps['modelo'].coef_[0], index=FEATURES).sort_values()
plt.figure(figsize=(8, 5))
coefs.plot(kind='barh', color=['tomato' if c > 0 else 'seagreen' for c in coefs])
plt.axvline(0, color='k')
plt.title("Coeficientes de la regresión logística\n(rojo = más riesgo de retención baja, verde = menos riesgo)")
plt.show()

In [ ]:
# 3) Relaciones lineales y no lineales: gráficos de dependencia parcial (PDP)
# Muestran cómo cambia la probabilidad de retención baja al variar UNA variable. Si no es una recta, el efecto es no lineal.
vars_num_top = [v for v in vars_top if v in VARS_NUM][:3]
Xte_f = Xte.astype(float)    # el PDP necesita columnas decimales (float)
print("Variables numéricas para el PDP:", vars_num_top)

fig, ax = plt.subplots(1, len(vars_num_top), figsize=(5 * len(vars_num_top), 4))
ax = np.atleast_1d(ax)
PartialDependenceDisplay.from_estimator(modelo_int, Xte_f, vars_num_top, ax=ax)
plt.suptitle(f"Dependencia parcial - {mejor_modelo}")
plt.tight_layout()
plt.show()

# ¿Un modelo no lineal rinde mejor que uno lineal?
auc_lineal = roc_auc_score(y_test, score_test['LoR'])
auc_no_lineal = roc_auc_score(y_test, score_test[mejor_modelo])
print(f"AUC regresión logística (lineal): {auc_lineal:.3f} | AUC {mejor_modelo}: {auc_no_lineal:.3f}")
print("Hay señales de relaciones no lineales o interacciones" if auc_no_lineal - auc_lineal > 0.02
      else "Las relaciones parecen principalmente lineales (el modelo no lineal no mejora de forma importante)")

In [ ]:
# 4) Interacción entre dos variables (PDP en 2 dimensiones)
if len(vars_num_top) >= 2:
    fig, ax = plt.subplots(figsize=(6, 5))
    PartialDependenceDisplay.from_estimator(modelo_int, Xte_f, [(vars_num_top[0], vars_num_top[1])], ax=ax)
    plt.title(f"Interacción: {vars_num_top[0]} y {vars_num_top[1]}")
    plt.show()

In [ ]:
# 5) Predicciones individuales con la regresión logística:
#    contribución de cada variable = coeficiente x valor estandarizado
pipe_lor = modelos_entrenados['LoR']
pre = pipe_lor[:2]                                    # solo imputar + escalar
coef = pipe_lor.named_steps['modelo'].coef_[0]
prob = pipe_lor.predict_proba(Xte)[:, 1]

casos = {'Mayor riesgo': int(np.argmax(prob)), 'Riesgo medio': int(np.argsort(prob)[len(prob) // 2]), 'Menor riesgo': int(np.argmin(prob))}
for nombre, i in casos.items():
    z = pre.transform(Xte.iloc[[i]])[0]
    contrib = pd.Series(z * coef, index=FEATURES).sort_values(key=np.abs, ascending=False).head(3)
    print(f"{nombre}: probabilidad de retención baja = {prob[i]*100:.1f}% | real = {'Retención baja' if y_test.iloc[i] == 1 else 'Normal'}")
    for var, v in contrib.items():
        print(f"    {var:35s} {'sube' if v > 0 else 'baja'} el riesgo ({v:+.2f})")

**Coherencia con la literatura:** compara las variables más importantes y la dirección de sus efectos (gráficos anteriores) con los antecedentes del tema. En general, la literatura sobre IA generativa y aprendizaje sugiere que el uso para **reemplazar el esfuerzo** (dependencia alta, mucha IA y poco estudio propio) se asocia con menor retención, mientras que el estudio activo y una buena base previa la favorecen. Si los resultados del proyecto coinciden con esto, apoyan esas ideas; si no, hay que discutir el porqué (el dataset es de un solo contexto y las relaciones son asociaciones, no causas).

### 5.5. Evaluación de sesgos y equidad

Revisamos si el modelo funciona igual para distintos grupos. En este dataset **no hay** sexo, edad, universidad ni nivel socioeconómico, por lo que usamos las variables disponibles:

* Área de carrera, año de estudio (equivale al avance académico), política institucional y habilidad de prompts.
* Suscripción paga (puede funcionar como indicador indirecto de recursos económicos).

Calculamos sensibilidad (recall), especificidad y tasas de error por grupo, usando el mejor modelo en el conjunto de prueba.

In [ ]:
grupos_test = df_sel.loc[Xte.index, ['Major_Category', 'Year_of_Study', 'Institutional_Policy',
                                      'Prompt_Engineering_Skill', 'Paid_Subscription']]
y_pred_mejor = pred_test[mejor_modelo]

def metricas_por_grupo(columna):
    filas = []
    for grupo in sorted(grupos_test[columna].unique(), key=str):
        m = (grupos_test[columna] == grupo).values
        yt, yp = y_test.values[m], y_pred_mejor[m]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        filas.append({
            'Grupo'          : grupo,
            'N'              : int(m.sum()),
            '% retención baja real': round(yt.mean() * 100, 1),
            'Sensibilidad'   : tp / (tp + fn) if (tp + fn) > 0 else np.nan,
            'Especificidad'  : tn / (tn + fp) if (tn + fp) > 0 else np.nan,
            'Tasa falsos positivos': fp / (fp + tn) if (fp + tn) > 0 else np.nan,
            'Tasa falsos negativos': fn / (fn + tp) if (fn + tp) > 0 else np.nan,
            'Accuracy'       : (tp + tn) / len(yt),
        })
    return pd.DataFrame(filas).set_index('Grupo')

resumen_equidad = []
for col in grupos_test.columns:
    tabla = metricas_por_grupo(col)
    print(f"===== {col} =====")
    display(tabla.round(3))
    dif_sens = tabla['Sensibilidad'].max() - tabla['Sensibilidad'].min()
    dif_esp  = tabla['Especificidad'].max() - tabla['Especificidad'].min()
    resumen_equidad.append({'Variable': col, 'Dif. sensibilidad': dif_sens, 'Dif. especificidad': dif_esp,
                            'Alerta (>0.10)': 'Sí' if max(dif_sens, dif_esp) > 0.10 else 'No',
                            'Grupo más pequeño (N)': int(tabla['N'].min())})

In [ ]:
# Resumen: diferencia máxima entre grupos. Una diferencia mayor a 0.10 es una señal de alerta.
display(pd.DataFrame(resumen_equidad).round(3))

f, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, col in zip(axes, ['Major_Category', 'Year_of_Study', 'Institutional_Policy']):
    metricas_por_grupo(col)[['Sensibilidad', 'Especificidad']].plot(kind='bar', ax=ax)
    ax.set_title(f"Sensibilidad y especificidad por {col}")
    ax.set_xlabel(""); ax.set_ylim(0, 1); ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

**Cómo leer esto:** si la sensibilidad es mucho menor en un grupo, el modelo "deja pasar" más estudiantes en riesgo de ese grupo. Si la tasa de falsos positivos es mayor, se marcaría injustamente a más estudiantes de ese grupo. Los grupos con pocos estudiantes (columna N) dan estimaciones poco confiables.

### 5.6. Evaluación de los objetivos de investigación

In [ ]:
# Resumen automático a partir de los resultados obtenidos
auc_mejor = roc_auc_score(y_test, score_test[mejor_modelo])
ic_inf = np.percentile(aucs_boot[mejor_modelo], 2.5)

print("HIPÓTESIS (sección 2.2 C)")
display(df_hip)

print("\nPREGUNTAS DE EVALUACIÓN")
print(f"1) ¿Se alcanzó el objetivo general? ROC-AUC en prueba = {auc_mejor:.3f} (meta > 0.70) ->",
      "SÍ" if auc_mejor > 0.70 else "PARCIALMENTE (no llega a la meta)")
print(f"2) ¿Se respondieron las preguntas de investigación? P1-P3 con las hipótesis H1-H3; P4 con los modelos; P5 con el clustering.")
n_apoyadas = (df_hip['Decisión'].str.contains('Se rechaza')).sum()
print(f"3) ¿Se contrastaron las hipótesis? Sí: {len(df_hip)} contrastadas, {n_apoyadas} apoyan H1 y {len(df_hip) - n_apoyadas} no se pudieron rechazar.")
print(f"4) ¿Se identificaron las variables relevantes? Sí. Las 5 principales: {vars_top}")
print(f"5) ¿Tiene utilidad científica el modelo? Su AUC (IC 95% desde {ic_inf:.3f}) es",
      "claramente mejor que el azar (0.5)." if ic_inf > 0.5 else "no se distingue claramente del azar.")
print("6) ¿Son consistentes con los antecedentes? Comparar las variables importantes y el signo de los efectos (5.4) con la literatura.")

### 5.7. Selección del modelo final

Se elige considerando **seis criterios a la vez** (no solo el AUC). Cada criterio se convierte a una escala de 0 a 1 y se combina con pesos:

| Criterio | Cómo se mide | Peso |
|----------|--------------|------|
| 1. Rendimiento predictivo | ROC-AUC en prueba | 30 % |
| 2. Capacidad de generalización | Brecha entrenamiento-prueba (menor = mejor) | 20 % |
| 3. Estabilidad | Desviación del AUC entre pliegues (menor = mejor) | 10 % |
| 4. Interpretabilidad | Puntaje 1-5 según el tipo de modelo | 15 % |
| 5. Complejidad computacional | Tiempo de entrenamiento (menor = mejor) | 10 % |
| 6. Aplicación práctica | Sensibilidad en prueba: detectar a los estudiantes en riesgo | 15 % |

> Los pesos son una decisión del equipo y se pueden cambiar.

In [ ]:
# Puntaje de interpretabilidad (1 = caja negra, 5 = muy fácil de explicar)
interpretabilidad = {'LoR': 5, 'LDA': 5, 'CART': 4, 'NB': 4, 'RF': 3, 'AdaBoost': 3,
                     'k-NN': 2, 'SVM': 2, 'GBM': 2, 'XGB': 2, 'LGBM': 2, 'CatBoost': 2}

crit = pd.DataFrame(index=comparacion.index)
crit['AUC prueba']         = tabla_test.set_index('Modelo')['ROC-AUC']
crit['Brecha']             = gen['Brecha entren.-prueba']
crit['Desv_AUC']           = comparacion['AUC_std']
crit['Interpretabilidad']  = [interpretabilidad[m.split(' (')[0]] for m in crit.index]
crit['Tiempo_s']           = comparacion['Tiempo_fit_s']
crit['Sensibilidad']       = tabla_test.set_index('Modelo')['Recall']

def a_escala(serie, mayor_es_mejor=True):
    rango = serie.max() - serie.min()
    if rango == 0:
        return pd.Series(1.0, index=serie.index)
    esc = (serie - serie.min()) / rango
    return esc if mayor_es_mejor else 1 - esc

pesos = {'AUC prueba': 0.30, 'Brecha': 0.20, 'Desv_AUC': 0.10, 'Interpretabilidad': 0.15, 'Tiempo_s': 0.10, 'Sensibilidad': 0.15}
mayor_mejor = {'AUC prueba': True, 'Brecha': False, 'Desv_AUC': False, 'Interpretabilidad': True, 'Tiempo_s': False, 'Sensibilidad': True}

puntaje = pd.DataFrame({c: a_escala(crit[c], mayor_mejor[c]) * pesos[c] for c in pesos})
crit['PUNTAJE FINAL'] = puntaje.sum(axis=1)
crit = crit.sort_values('PUNTAJE FINAL', ascending=False)
crit.round(3)

In [ ]:
nombre_final = crit.index[0]
modelo_final = modelos_entrenados[nombre_final]
y_pred_final = modelo_final.predict(Xte)

print(f"MODELO FINAL SELECCIONADO: {nombre_final}")
print(f"  - Rendimiento  : AUC en prueba = {crit.loc[nombre_final, 'AUC prueba']:.3f}")
print(f"  - Generalización: brecha entrenamiento-prueba = {crit.loc[nombre_final, 'Brecha']:.3f}")
print(f"  - Estabilidad  : desviación del AUC entre pliegues = {crit.loc[nombre_final, 'Desv_AUC']:.3f}")
print(f"  - Interpretabilidad: {crit.loc[nombre_final, 'Interpretabilidad']:.0f} de 5")
print(f"  - Sensibilidad : detecta {crit.loc[nombre_final, 'Sensibilidad']*100:.1f}% de los estudiantes con retención baja")
print(f"  - Mejor AUC puro fue: {tabla_test.loc[0, 'Modelo']}")
print()
print(classification_report(y_test, y_pred_final, target_names=['Normal', 'Retención baja']))

In [ ]:
# Guardamos el modelo final para poder reutilizarlo
import joblib
joblib.dump({'modelo': modelo_final, 'variables': FEATURES, 'umbral_percentil': UMBRAL_PERCENTIL}, 'modelo_final_retencion.pkl')
print("Modelo guardado en: modelo_final_retencion.pkl")

## Conclusiones y limitaciones

**Conclusiones** (verificar con los resultados de cada sección):
1. Se construyó un flujo completo CRISP-DM: comprensión del negocio, de los datos, preparación, modelado y evaluación.
2. Las hipótesis H1 a H5 se contrastaron en la sección 2.2 (C) y se resumen en la 5.6.
3. Se compararon 12 algoritmos y 3 optimizados con las mismas particiones y métricas. El modelo final se eligió con seis criterios, no solo por el AUC (sección 5.7).
4. El análisis no supervisado (secciones 4.2 y 5.1 B) permite describir perfiles de estudiantes según su uso de IA.

**Limitaciones:**
* Los resultados muestran **asociaciones, no causas**.
* `Perceived_AI_Dependency` y `Anxiety_Level_During_Exams` son autorreportadas.
* La variable objetivo depende del umbral elegido (30 % más bajo). Se puede probar con otros porcentajes cambiando `UMBRAL_PERCENTIL`.
* No hay validación externa ni variables como sexo, edad o nivel socioeconómico para un análisis de equidad completo.
* El origen del dataset no está bien documentado; conviene confirmarlo antes de generalizar.

**Trabajo futuro:** validar con datos de otra universidad, probar otros umbrales y modelos de regresión sobre el puntaje continuo, y usar SHAP para explicaciones individuales más detalladas.